In [3]:
import numpy as np

In [4]:
data = np.load("dataset.npz")

X_train = data["X_train"]
y_train = data["y_train"]
X_val = data["X_val"]
y_val = data["y_val"]
X_test = data["X_test"]
y_test = data["y_test"]

In [35]:
from sklearn.base import BaseEstimator, RegressorMixin
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

class IdentityRegressor(RegressorMixin, BaseEstimator):
    def fit(self, X, y=None):
        # Aucun paramètre à apprendre
        return self

    def predict(self, X):
        # On suppose que X contient une seule colonne
        return np.asarray(X)


model = IdentityRegressor()
model.fit(X_train, y_train)
y_pred_identity = model.predict(X_test)

In [36]:
from sklearn.linear_model import LinearRegression


# Création et entraînement
model =make_pipeline(
    StandardScaler(),
    LinearRegression(),
)
model.fit(X_train, y_train)

# Prédictions
y_pred_linear = model.predict(X_test)

# Évaluation
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print(f"MAE  : {mae:.2f}")
print(f"RMSE : {rmse:.2f}")

MAE  : 1545.46
RMSE : 2116.86


In [37]:
import pandas as pd

predictions = {
    "Identité": y_pred_identity,
    "Régression linéaire": y_pred_linear,
}

results = pd.DataFrame({
    name: {
        "MAE": mean_absolute_error(y_test, pred),
        "RMSE": np.sqrt(mean_squared_error(y_test, pred)),
    }
    for name, pred in predictions.items()
}).T

for metric in ["MAE", "RMSE"]:
    results[f"Gain {metric} (%)"] = (
        100 * (1 - results[metric] / results.loc["Identité", metric])
    )

display(results.round(2))

,MAE,RMSE,Gain MAE (%),Gain RMSE (%)
Identité,2486.28,3626.02,0.00,0.00
Régression linéaire,1545.46,2116.86,37.84,41.62


In [38]:
import numpy as np
import pandas as pd

from sklearn.model_selection import TimeSeriesSplit


cv = TimeSeriesSplit(n_splits=5)
rows = []

# Position de la consommation utilisée par le modèle identité dans X
consumption_col = 0

for window, (train_idx, val_idx) in enumerate(cv.split(X_train), start=1):
    X_fit = X_train[train_idx]
    y_fit = y_train[train_idx]

    X_val = X_train[val_idx]
    y_val = y_train[val_idx]

    model = LinearRegression()
    model.fit(X_fit, y_fit)

    # ravel() donne des vecteurs de forme (n,)
    pred_linear = model.predict(X_val).ravel()
    pred_identity = X_val[:, consumption_col]
    target = y_val.ravel()

    mae_identity = mean_absolute_error(target, pred_identity)
    mae_linear = mean_absolute_error(target, pred_linear)

    rows.append({
        "Fenêtre": window,
        "MAE identité": mae_identity,
        "MAE régression": mae_linear,
        "Gain MAE (%)": (
            100 * (1 - mae_linear / mae_identity)
            if mae_identity > 0 else np.nan
        ),
    })

results = pd.DataFrame(rows).set_index("Fenêtre")
display(results.round(2))

ValueError: Found input variables with inconsistent numbers of samples: [29616, 617]